# Detector de huevos v3: huevos sanos en fotos reales, fuera del montaje

**Por qué hay un v3.** `v2` acierta el test original (mAP50-95 0.97) y las sintéticas, pero con **fotos reales de otras fuentes** (cocinas, manos, mesas) marca como `Crack` casi todos los huevos sanos: todas las fotos reales de `Intact` del dataset son del mismo montaje. Las sintéticas de `v2` ayudaron, pero no bastan. `v3` añade **fotos reales de huevos sanos y rajados de otra fuente**.

**Qué hace este notebook** (Colab con GPU, *Entorno de ejecución > Ejecutar todo*):
1. Setup: GPU, `ultralytics` (misma versión que `v2`), Drive y dataset.
2–3. Reconstruye el dataset de `v2` (original + sintéticas) con las mismas funciones y semillas.
4. Fotos reales: descarga un dataset público (273 huevos sanos y 695 defectuosos) y pone las cajas con **YOLO-World** (detector de vocabulario abierto, clase `egg`), sin usar `v2`. Separa train/valid/test por bloques consecutivos para que fotos casi iguales no queden en train y test.
5. Hojas de auditoría de esas cajas.
6. Entrena `v3` partiendo de `v2/best.pt`.
7. Compara `v2` y `v3`: test original, sintéticas, fotos reales públicas y un **test independiente de Wikimedia Commons** (fotos de otra fuente que no se usan para entrenar ni para elegir). Solo se elige `v3` si no empeora el test original.
8–9. Exporta a `.tflite` NHWC (FP32 e INT8, igual que `v2`), verifica y guarda `resumen.json`.

Nada se sobrescribe: el run nuevo se llama `v3` y el export va a `exports/v3`. Si Colab se desconecta, vuelve a *Ejecutar todo*: lo terminado se salta y el entrenamiento se reanuda.

## 1. Setup
Instala `ultralytics` 8.4.161 (la versión con la que se entrenó `v2`), monta Drive, busca la carpeta `eggs_v2` y el `best.pt` de `v2`, y descomprime el dataset. Si no tienes permiso de escritura en la carpeta del equipo, los resultados se guardan en `MyDrive/eggs_v3`.

In [ ]:
%pip install -q "ultralytics==8.4.161"

import json
import random
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import requests
import yaml
from google.colab import drive
import ultralytics
import torch

UA = {'User-Agent': 'eggs-detector-eval/1.0 (academic project; https://github.com/dportilla219/eggs-detector)'}


def bajar(url, destino, intentos=4):
    # Descarga una imagen (con reintentos, por si Wikimedia limita la velocidad) y comprueba que se pueda abrir.
    import time
    from PIL import Image
    if destino.exists():
        return True
    for k in range(intentos):
        try:
            r = requests.get(url, headers=UA, timeout=60)
            if r.ok:
                destino.write_bytes(r.content)
                Image.open(destino).verify()
                return True
        except Exception:
            destino.unlink(missing_ok=True)
        time.sleep(2 * (k + 1))
    print('No se pudo descargar (se omite):', url)
    return False

drive.mount('/content/drive')
ultralytics.checks()
assert torch.cuda.is_available(), ('Este notebook necesita GPU: Entorno de ejecución > Cambiar tipo de entorno de '
                                   'ejecución > T4 GPU, y vuelve a ejecutar todo.')

MYDRIVE = Path('/content/drive/MyDrive')
ZIPS = ['eggs_v2_parte1_train.zip', 'eggs_v2_parte2_train.zip',
        'eggs_v2_parte3_train.zip', 'eggs_v2_parte4_valid_test.zip']

# Carpeta con el dataset: la del equipo si está en tu Drive (o como acceso directo); si no, MyDrive/eggs_v2
hallados = [p for patron in (f'eggs_v2/{ZIPS[0]}', f'*/{ZIPS[0]}', f'*/*/{ZIPS[0]}') for p in sorted(MYDRIVE.glob(patron))]
DRIVE_DIR = hallados[0].parent if hallados else MYDRIVE / 'eggs_v2'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)


def es_v2(p):
    # True si el .pt es el detector v2 de huevos (y no otro modelo con el mismo nombre de archivo).
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        nombres = getattr(ck.get('ema') or ck.get('model'), 'names', None)
        return nombres == {0: 'Crack', 1: 'Intact'} and ck.get('train_args', {}).get('name') == 'v2'
    except Exception:
        return False


def buscar_v2():
    candidatos = [DRIVE_DIR / 'runs/v2/weights/best.pt', DRIVE_DIR / 'exports/v2/eggs_v2.pt', *sorted(DRIVE_DIR.glob('*.pt'))]
    return next((p for p in candidatos if p.exists() and es_v2(p)), None)


def reensamblar():
    # Une los trozos "<archivo>.parte1de2", "<archivo>.parte2de2"... cuando están todos (subidas de menos de 10 MB).
    for f in sorted(DRIVE_DIR.glob('*.parte1de*')):
        nombre, total = f.name.rsplit('.parte1de', 1)
        partes = [DRIVE_DIR / f'{nombre}.parte{i}de{total}' for i in range(1, int(total) + 1)]
        if (DRIVE_DIR / nombre).exists() or not all(p.exists() for p in partes):
            continue
        with open(DRIVE_DIR / nombre, 'wb') as out:
            for p in partes:
                out.write(p.read_bytes())
        if nombre.endswith('.zip'):
            assert zipfile.ZipFile(DRIVE_DIR / nombre).testzip() is None, f'{nombre} quedó dañado al unirlo'
        print('Unido:', nombre)


def que_falta():
    return [z for z in ZIPS if not (DRIVE_DIR / z).exists()] + ([] if buscar_v2() else ['best.pt (pesos de v2)'])


reensamblar()
faltan = que_falta()
while faltan:
    # Primera vez: se suben desde el PC y quedan guardados en tu Drive para las siguientes ejecuciones.
    from google.colab import files
    print('Faltan en tu Drive:', faltan)
    print('Pulsa "Elegir archivos" y selecciona en tu carpeta Descargas los 4 zips eggs_v2_parte*.zip y best.pt '
          '(con Ctrl puedes elegir varios a la vez). También sirven en trozos .parteNdeM.')
    for nombre, datos in files.upload().items():
        nombre = re.sub(r' \(\d+\)(?=\.|$)', '', nombre)  # "best (1).pt" -> best.pt
        (DRIVE_DIR / nombre).write_bytes(datos)
        print('Guardado en tu Drive:', DRIVE_DIR / nombre)
    reensamblar()
    faltan = que_falta()
BASE_PT = buscar_v2()


def escribible(d):
    try:
        d.mkdir(parents=True, exist_ok=True)
        (d / '.prueba').write_text('ok')
        (d / '.prueba').unlink()
        return True
    except OSError:
        return False


OUT_DIR = DRIVE_DIR if escribible(DRIVE_DIR / 'runs') else MYDRIVE / 'eggs_v3'
RUNS_DIR, EXPORTS_DIR = OUT_DIR / 'runs', OUT_DIR / 'exports'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path('/content/eggs_v2')          # dataset original
V2_DIR = Path('/content/eggs_v2s')           # dataset original + sintéticas (igual que en v2)
PUB_DIR = Path('/content/publico')           # fotos reales públicas en formato YOLO
BASE_RUN, NEW_RUN = 'v2', 'v3'
CLASSES = ['Crack', 'Intact']
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if not (DATA_DIR / 'train').exists():
    for n in ['eggs_v2_parte1_train.zip', 'eggs_v2_parte2_train.zip',
              'eggs_v2_parte3_train.zip', 'eggs_v2_parte4_valid_test.zip']:
        with zipfile.ZipFile(DRIVE_DIR / n) as zf:
            zf.extractall(DATA_DIR)
        print('OK', n)
with open(DATA_DIR / 'data.yaml', 'w') as f:
    yaml.safe_dump({'path': str(DATA_DIR), 'train': str(DATA_DIR / 'train/images'),
                    'val': str(DATA_DIR / 'valid/images'), 'test': str(DATA_DIR / 'test/images'),
                    'nc': 2, 'names': CLASSES}, f, sort_keys=False)

libre, total = torch.cuda.mem_get_info()
print(f'GPU: {torch.cuda.get_device_name(0)}, libre {libre / 1e9:.1f} de {total / 1e9:.1f} GB')
print('Pesos de v2:', BASE_PT)
print('Resultados en:', OUT_DIR)

## 2. Funciones de imágenes sintéticas
Copia literal de `eggs_v2.ipynb` (celda 2). Solo define funciones.

In [ ]:
import random
import shutil
from collections import Counter

import cv2
import numpy as np


def fuente(stem):
    """'montaje' = fotos del montaje de fondo gris (todas las Intact y parte de las Crack)."""
    return 'montaje' if stem.startswith('ec_egg') else 'otras'


def cargar_split(split_dir):
    """Lista de dicts {img, cls, box (x1,y1,x2,y2 en px), fuente} de las imágenes originales (sin _dup)."""
    items = []
    for img in sorted((split_dir / 'images').iterdir()):
        if img.suffix.lower() not in IMG_EXTS or '_dup' in img.stem:
            continue
        lines = (split_dir / 'labels' / f'{img.stem}.txt').read_text().split('\n')
        parts = lines[0].split()
        if len(parts) != 5:
            continue
        h, w = cv2.imread(str(img)).shape[:2]
        c, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:])
        box = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        items.append({'img': img, 'cls': c, 'box': box, 'fuente': fuente(img.stem), 'size': (w, h)})
    return items


def recortar_huevo(item):
    """Recorta el huevo con una máscara elíptica suavizada (los huevos son casi elipses)."""
    img = cv2.imread(str(item['img']))
    x1, y1, x2, y2 = [int(round(v)) for v in item['box']]
    x1, y1 = max(x1, 0), max(y1, 0)
    x2, y2 = min(x2, img.shape[1]), min(y2, img.shape[0])
    crop = img[y1:y2, x1:x2].copy()
    h, w = crop.shape[:2]
    mask = np.zeros((h, w), np.float32)
    cv2.ellipse(mask, (w // 2, h // 2), (max(w // 2 - 1, 1), max(h // 2 - 1, 1)), 0, 0, 360, 1.0, -1)
    k = max(3, (min(w, h) // 25) | 1)
    mask = cv2.GaussianBlur(mask, (k, k), 0)
    return crop, mask


def variar_huevo(crop, rng):
    """Volteo aleatorio y, a veces, 'blanquear' el huevo (hay huevos blancos sanos en la vida real)."""
    if rng.random() < 0.5:
        crop = crop[:, ::-1]
    if rng.random() < 0.3:
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= rng.uniform(0.1, 0.4)
        hsv[..., 2] = np.clip(hsv[..., 2] * rng.uniform(1.1, 1.4) + 20, 0, 255)
        crop = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    return np.ascontiguousarray(crop)


def pegar(bg, crop, mask, cx, cy, w, h):
    """Pega el huevo (w x h px) centrado en (cx, cy). Devuelve la caja visible o None si queda muy cortado."""
    w, h = max(int(w), 4), max(int(h), 4)
    crop = cv2.resize(crop, (w, h), interpolation=cv2.INTER_AREA if w < crop.shape[1] else cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (w, h))[..., None]
    H, W = bg.shape[:2]
    x1, y1 = int(cx - w / 2), int(cy - h / 2)
    bx1, by1, bx2, by2 = max(x1, 0), max(y1, 0), min(x1 + w, W), min(y1 + h, H)
    if (bx2 - bx1) * (by2 - by1) < 0.85 * w * h:
        return None
    roi = bg[by1:by2, bx1:bx2].astype(np.float32)
    c = crop[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1].astype(np.float32)
    m = mask[by1 - y1:by2 - y1, bx1 - x1:bx2 - x1]
    bg[by1:by2, bx1:bx2] = (c * m + roi * (1 - m)).astype(np.uint8)
    return bx1, by1, bx2, by2


def fondo_procedural(rng, W, H):
    """Fondos variados (color liso con ruido, degradado o textura) que no pertenecen a ninguna clase."""
    tipo = rng.randrange(3)
    c1 = np.array([rng.randrange(256) for _ in range(3)], np.float32)
    if tipo == 0:
        bg = np.ones((H, W, 3), np.float32) * c1
    elif tipo == 1:
        c2 = np.array([rng.randrange(256) for _ in range(3)], np.float32)
        t = np.linspace(0, 1, W if rng.random() < 0.5 else H, dtype=np.float32)
        t = t[None, :, None] if len(t) == W else t[:, None, None]
        bg = np.broadcast_to(c1 * (1 - t) + c2 * t, (H, W, 3)).copy()
    else:
        ruido = np.random.default_rng(rng.randrange(1 << 30)).random((H // 16 + 1, W // 16 + 1, 1)).astype(np.float32)
        ruido = cv2.resize(ruido, (W, H), interpolation=cv2.INTER_CUBIC)[..., None]
        bg = c1 * (0.6 + 0.8 * ruido)
    bg += np.random.default_rng(rng.randrange(1 << 30)).normal(0, rng.uniform(2, 12), (H, W, 3)).astype(np.float32)
    return np.clip(bg, 0, 255).astype(np.uint8)


def mira_verde(img, rng):
    """La mira verde del montaje, dibujada en imágenes de ambas clases para que no sea pista de Intact."""
    H, W = img.shape[:2]
    cx, cy = int(W * rng.uniform(0.4, 0.6)), int(H * rng.uniform(0.4, 0.6))
    color = (int(rng.uniform(80, 140)), int(rng.uniform(170, 230)), int(rng.uniform(80, 140)))
    cv2.line(img, (0, cy), (W, cy), color, 1)
    cv2.line(img, (cx, 0), (cx, H), color, 1)


def retocar(img, cajas, rng):
    """Mira verde en ambas clases y, a veces, bajar la resolución de toda la imagen para que la nitidez
    tampoco delate la clase (los huevos Intact vienen de fotos de 224 px)."""
    if rng.random() < 0.4:
        mira_verde(img, rng)
    if rng.random() < 0.5:
        H, W = img.shape[:2]
        f = rng.uniform(224, 320) / max(H, W)
        if f < 1:
            img = cv2.resize(img, (max(int(W * f), 1), max(int(H * f), 1)), interpolation=cv2.INTER_AREA)
            cajas = [(c, x1 * f, y1 * f, x2 * f, y2 * f) for c, x1, y1, x2, y2 in cajas]
    return img, cajas


def guardar(out_dir, nombre, img, cajas):
    """cajas: lista de (cls, x1, y1, x2, y2) en px -> .jpg + .txt YOLO."""
    H, W = img.shape[:2]
    cv2.imwrite(str(out_dir / 'images' / f'{nombre}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    lineas = [f'{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'
              for c, x1, y1, x2, y2 in cajas]
    (out_dir / 'labels' / f'{nombre}.txt').write_text('\n'.join(lineas) + '\n')


def intercambio(fondo_item, huevo_item, rng):
    """Tapa el huevo del fondo con el otro huevo (un 12-25 % más grande para cubrirlo entero)."""
    bg = cv2.imread(str(fondo_item['img']))
    crop, mask = recortar_huevo(huevo_item)
    crop = variar_huevo(crop, rng)
    x1, y1, x2, y2 = fondo_item['box']
    ch, cw = crop.shape[:2]
    s = max((x2 - x1) / cw, (y2 - y1) / ch) * rng.uniform(1.12, 1.25)
    return bg, pegar(bg, crop, mask, (x1 + x2) / 2, (y1 + y2) / 2, cw * s, ch * s)


def procedural(huevos, rng):
    """1 o 2 huevos sobre un fondo procedural, con tamaño y posición al azar."""
    W, H = rng.choice([(640, 640), (640, 480), (480, 640), (288, 640)])
    bg = fondo_procedural(rng, W, H)
    cajas = []
    for _ in range(2 if rng.random() < 0.3 else 1):
        item = rng.choice(huevos)
        crop, mask = recortar_huevo(item)
        crop = variar_huevo(crop, rng)
        ch, cw = crop.shape[:2]
        s = rng.uniform(0.25, 0.7) * min(W, H) / max(ch, cw)
        w, h = cw * s, ch * s
        for _intento in range(10):
            cx, cy = rng.uniform(w / 2, W - w / 2), rng.uniform(h / 2, H - h / 2)
            if all(cx + w / 2 < a or cx - w / 2 > c or cy + h / 2 < b or cy - h / 2 > d for _, a, b, c, d in cajas):
                caja = pegar(bg, crop, mask, cx, cy, w, h)
                if caja:
                    cajas.append((item['cls'], *caja))
                break
    return bg, cajas


def generar_sinteticas(split_dir, out_dir, n_a, n_b, n_p, seed, prefijo):
    """A: Intact del montaje sobre fondos de otras fuentes. B: Crack de otras fuentes sobre el montaje.
    P: huevos de ambas clases sobre fondos procedurales. Solo usa huevos y fondos del mismo split."""
    rng = random.Random(seed)
    items = cargar_split(split_dir)
    intact = [i for i in items if i['cls'] == 1]
    crack_otras = [i for i in items if i['cls'] == 0 and i['fuente'] == 'otras']
    montaje = [i for i in items if i['fuente'] == 'montaje']
    (out_dir / 'images').mkdir(parents=True, exist_ok=True)
    (out_dir / 'labels').mkdir(parents=True, exist_ok=True)
    conteo = Counter()
    tareas = [('A', n_a, crack_otras, intact), ('B', n_b, montaje, crack_otras)]
    for tipo, n, fondos, huevos in tareas:
        hechos = 0
        orden = (huevos * (n // len(huevos) + 1))[:n] if tipo == 'A' else [rng.choice(huevos) for _ in range(n)]
        for k, huevo in enumerate(orden):
            for _intento in range(5):
                bg, caja = intercambio(rng.choice(fondos), huevo, rng)
                if caja:
                    break
            if not caja:
                continue
            bg, cajas = retocar(bg, [(huevo['cls'], *caja)], rng)
            guardar(out_dir, f'{prefijo}_{tipo}_{k:05d}', bg, cajas)
            conteo[(tipo, CLASSES[huevo['cls']])] += 1
            hechos += 1
    # mitad Intact / mitad Crack para que el fondo procedural no favorezca a ninguna clase
    pools = [intact, [i for i in items if i['cls'] == 0]]
    for k in range(n_p):
        bg, cajas = procedural(pools[k % 2], rng)
        if not cajas:
            continue
        bg, cajas = retocar(bg, cajas, rng)
        guardar(out_dir, f'{prefijo}_P_{k:05d}', bg, cajas)
        for c, *_ in cajas:
            conteo[('P', CLASSES[c])] += 1
    return conteo

## 3. Dataset de v2 (original + sintéticas)
Copia literal de `eggs_v2.ipynb` (celda 3): mismas semillas, así que sale el mismo dataset con el que se entrenó `v2`. `v3` lo sigue viendo para no olvidar lo que ya sabe.

In [ ]:
import pandas as pd

if V2_DIR.exists():
    shutil.rmtree(V2_DIR)

def copiar_originales(split, destino):
    for sub in ['images', 'labels']:
        (destino / sub).mkdir(parents=True, exist_ok=True)
    for img in (DATA_DIR / split / 'images').iterdir():
        if img.suffix.lower() in IMG_EXTS and '_dup' not in img.stem:
            shutil.copy2(img, destino / 'images' / img.name)
            shutil.copy2(DATA_DIR / split / 'labels' / f'{img.stem}.txt', destino / 'labels' / f'{img.stem}.txt')

conteos = {}
copiar_originales('train', V2_DIR / 'train')
conteos['train'] = generar_sinteticas(DATA_DIR / 'train', V2_DIR / 'train', 1228, 700, 700, seed=0, prefijo='syn_train')
copiar_originales('valid', V2_DIR / 'valid')
conteos['valid'] = generar_sinteticas(DATA_DIR / 'valid', V2_DIR / 'valid', 356, 200, 200, seed=1, prefijo='syn_valid')
conteos['test_synth'] = generar_sinteticas(DATA_DIR / 'test', V2_DIR / 'test_synth', 168, 168, 200, seed=2, prefijo='syn_test')

def escribir_yaml(nombre, **splits):
    p = V2_DIR / nombre
    with open(p, 'w') as f:
        yaml.safe_dump({'path': str(V2_DIR), **splits, 'nc': 2, 'names': CLASSES}, f, sort_keys=False)
    return p

DATA_V2 = escribir_yaml('data.yaml', train=str(V2_DIR / 'train/images'), val=str(V2_DIR / 'valid/images'),
                        test=str(DATA_DIR / 'test/images'))
DATA_TEST_ALL = escribir_yaml('test_all.yaml', train=str(V2_DIR / 'train/images'),
                              val=[str(DATA_DIR / 'test/images'), str(V2_DIR / 'test_synth/images')])

def resumen(split_dir):
    c = {}
    for lp in (split_dir / 'labels').glob('*.txt'):
        for line in lp.read_text().splitlines():
            if line.strip():
                k = CLASSES[int(line.split()[0])]
                c[k] = c.get(k, 0) + 1
    return c

filas = {s: resumen(V2_DIR / s) for s in ['train', 'valid', 'test_synth']}
display(pd.DataFrame(filas).T.assign(ratio=lambda d: (d['Crack'] / d['Intact']).round(2)))
print({s: dict(c) for s, c in conteos.items()})

## 4. Fotos reales de otra fuente, con cajas de YOLO-World
Dataset público [Egg-Defect-Detection](https://github.com/dakshkathuria346-gif/Egg-Defect-Detection): fotos de celular de huevos sanos (`non defective`) y defectuosos (`defective`), sin cajas. Las cajas las pone **YOLO-World** con la clase `egg`, un detector de vocabulario abierto que no depende de lo que aprendió `v2`.

Filtros, para no enseñar etiquetas malas:
- una caja cuenta si su confianza es ≥ `CONF_OK` y tiene forma y tamaño de huevo;
- se descarta la imagen si hay alguna caja dudosa (posible huevo sin etiquetar);
- sanos: 1 a 6 huevos por foto. Defectuosos: exactamente 1 huevo (si hay varios no sabemos cuál está rajado).

Split por bloques consecutivos (70/15/15) dentro de cada carpeta, porque el mismo huevo aparece en fotos seguidas. Los sanos de train se repiten 3 veces para que pesen frente a los ~5.000 del dataset original.

In [ ]:
from ultralytics import YOLO, YOLOWorld

PUB_SRC = Path('/content/publico_src')
if not PUB_SRC.exists():
    !git clone -q --depth 1 https://github.com/dakshkathuria346-gif/Egg-Defect-Detection {PUB_SRC}

CONF_OK, CONF_DUDA = 0.30, 0.15
world = YOLOWorld('yolov8l-worldv2.pt')
world.set_classes(['egg'])


def orden_natural(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', p.name)]


def iou_px(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0


def forma_de_huevo(b, W, H):
    w, h = b[2] - b[0], b[3] - b[1]
    return 0.003 <= w * h / (W * H) <= 0.9 and 0.45 <= w / max(h, 1e-6) <= 2.2


def etiquetar(carpeta, clase, origen='pub'):
    files = sorted([p for p in Path(carpeta).iterdir() if p.suffix.lower() in IMG_EXTS], key=orden_natural)
    filas = []
    for f, r in zip(files, world.predict([str(p) for p in files], conf=CONF_DUDA, iou=0.5, imgsz=640,
                                          stream=True, verbose=False)):
        H, W = r.orig_shape
        cajas = list(zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()))
        buenas = [b for b, c in cajas if c >= CONF_OK and forma_de_huevo(b, W, H)]
        dudosas = [b for b, c in cajas if not (c >= CONF_OK and forma_de_huevo(b, W, H))
                   and all(iou_px(b, g) < 0.3 for g in buenas)]
        ok = not dudosas and (1 <= len(buenas) <= 6 if clase == 1 else len(buenas) == 1)
        filas.append({'file': f, 'clase': clase, 'W': W, 'H': H, 'cajas': buenas, 'ok': ok, 'origen': origen})
    return filas


# Fotos de huevos sanos de Wikimedia Commons (licencias libres) elegidas a mano SOLO para entrenar:
# blancos, marrones, verdes, en mano, en cartón, en taza... Ninguna es del mismo autor que las del test de Commons.
COMMONS_TRAIN = [
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b7/2_Huevos_de_distintas_especies_de_gallinas_21.jpg/960px-2_Huevos_de_distintas_especies_de_gallinas_21.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 Huevos de distintas especies de gallinas 21.jpg",
  "CC BY-SA 4.0",
  "MONUMENTA",
  292
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/86/2_Huevos_de_distintas_especies_de_gallinas_7.jpg/960px-2_Huevos_de_distintas_especies_de_gallinas_7.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 Huevos de distintas especies de gallinas 7.jpg",
  "CC BY-SA 4.0",
  "MONUMENTA",
  293
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ec/2_Huevos_de_distintas_especies_de_gallinas.jpg/960px-2_Huevos_de_distintas_especies_de_gallinas.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 Huevos de distintas especies de gallinas.jpg",
  "CC BY-SA 4.0",
  "MONUMENTA",
  294
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5b/20170602-AMS-PJK-0101_%2834890669702%29.jpg/960px-20170602-AMS-PJK-0101_%2834890669702%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:20170602-AMS-PJK-0101 (34890669702).jpg",
  "Public domain",
  "U.S. Department of Agriculture\n\nPreston Keres/Office of Communications-Photograp",
  298
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/23-365_-_Flickr_-_eren_%28sea-prairie%29.jpg/960px-23-365_-_Flickr_-_eren_%28sea-prairie%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:23-365 - Flickr - eren (sea-prairie).jpg",
  "CC BY-SA 2.0",
  "eren {sea+prairie}",
  301
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1f/A_Clutch_of_White_Bird_Eggs_Nestled_in_Dried_Grass.jpg/960px-A_Clutch_of_White_Bird_Eggs_Nestled_in_Dried_Grass.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:A Clutch of White Bird Eggs Nestled in Dried Grass.jpg",
  "CC BY-SA 4.0",
  "A S M Jobaer",
  312
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/69/An_Egg_%2832970626574%29.jpg/960px-An_Egg_%2832970626574%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:An Egg (32970626574).jpg",
  "CC BY 2.0",
  "NIAID",
  330
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5e/An_Egg_%2833684416751%29.jpg/960px-An_Egg_%2833684416751%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:An Egg (33684416751).jpg",
  "CC BY 2.0",
  "NIAID",
  331
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/9/95/Anne_Biermann_eier.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Anne Biermann eier.jpg",
  "Public domain",
  "aenne biermann",
  335
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/45/At_London_2025_112.jpg/960px-At_London_2025_112.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:At London 2025 112.jpg",
  "CC BY-SA 4.0",
  "Photograph by Mike Peel (www.mikepeel.net).",
  337
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/ad/At_London_2025_113.jpg/960px-At_London_2025_113.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:At London 2025 113.jpg",
  "CC BY-SA 4.0",
  "Photograph by Mike Peel (www.mikepeel.net).",
  338
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6d/Barred_Plymouth_Rock_egg_%28cropped%29.jpg/960px-Barred_Plymouth_Rock_egg_%28cropped%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Barred Plymouth Rock egg (cropped).jpg",
  "CC BY-SA 3.0",
  "",
  343
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6f/Barred_Plymouth_Rock_egg.jpg/960px-Barred_Plymouth_Rock_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Barred Plymouth Rock egg.jpg",
  "CC BY-SA 3.0",
  "",
  344
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/52/Blue_hen_eggs.jpg/960px-Blue_hen_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Blue hen eggs.jpg",
  "CC BY-SA 4.0",
  "Secretlondon",
  346
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/14/Boiled-Egg-spoons-with-egg.jpg/960px-Boiled-Egg-spoons-with-egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Boiled-Egg-spoons-with-egg.jpg",
  "CC BY-SA 4.0",
  "Grenadille",
  350
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Braunes_H%C3%BChnerei.JPG/960px-Braunes_H%C3%BChnerei.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Braunes Hühnerei.JPG",
  "CC BY-SA 3.0",
  "Fiver, der Hellseher",
  351
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/0f/Brown_Egg_%2818423463472%29.jpg/960px-Brown_Egg_%2818423463472%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Brown Egg (18423463472).jpg",
  "CC BY-SA 2.0",
  "Willis Lam",
  352
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/14/Chicken%27s_egg.jpg/960px-Chicken%27s_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken's egg.jpg",
  "CC BY-SA 4.0",
  "Chris Prince Udochukwu Njoku",
  374
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/79/Comparativa_huevo_de_gallina_y_de_avestruz.jpg/960px-Comparativa_huevo_de_gallina_y_de_avestruz.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Comparativa huevo de gallina y de avestruz.jpg",
  "CC BY-SA 4.0",
  "Kbemcap",
  379
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c8/Country_chicken_eggs.jpg/960px-Country_chicken_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Country chicken eggs.jpg",
  "CC BY-SA 4.0",
  "சு.பத்மா",
  380
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Country_Eggs_in_Salem.jpg/960px-Country_Eggs_in_Salem.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Country Eggs in Salem.jpg",
  "CC BY-SA 4.0",
  "Thamizhpparithi Maari",
  381
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/14/Egg_High_Key_%285452144772%29.jpg/960px-Egg_High_Key_%285452144772%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg High Key (5452144772).jpg",
  "CC BY 2.0",
  "Nolan Williamson from Atlanta, GA, United States of America",
  394
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a3/Egg_KyakOo.jpg/960px-Egg_KyakOo.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg KyakOo.jpg",
  "CC BY-SA 4.0",
  "SarKaLay စာကလေး",
  397
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/4/44/Egg_stand.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Egg stand.jpg",
  "CC BY-SA 2.0",
  "☈ASCAL",
  400
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/53/Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_01.jpg/960px-Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_01.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs in a plastic bowl gotten from Noiler Birds 01.jpg",
  "CC BY-SA 4.0",
  "Samsule2",
  413
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/36/Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_04.jpg/960px-Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_04.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs in a plastic bowl gotten from Noiler Birds 04.jpg",
  "CC BY-SA 4.0",
  "Samsule2",
  416
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/45/Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_05.jpg/960px-Eggs_in_a_plastic_bowl_gotten_from_Noiler_Birds_05.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs in a plastic bowl gotten from Noiler Birds 05.jpg",
  "CC BY-SA 4.0",
  "Samsule2",
  417
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/9/9a/Ei01.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Ei01.jpg",
  "Public domain",
  "No machine-readable author provided. Acf assumed (based on copyright claims).",
  424
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/Ei_und_H%C3%BChnerfeder_2.jpg/960px-Ei_und_H%C3%BChnerfeder_2.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei und Hühnerfeder 2.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  425
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7b/Ei_und_H%C3%BChnerfeder_5a.jpg/960px-Ei_und_H%C3%BChnerfeder_5a.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei und Hühnerfeder 5a.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  428
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d8/Ei_und_H%C3%BChnerfeder_5b.jpg/960px-Ei_und_H%C3%BChnerfeder_5b.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei und Hühnerfeder 5b.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  429
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4a/Eier_in_Eierkarton_5.jpg/960px-Eier_in_Eierkarton_5.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eier in Eierkarton 5.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  436
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8e/Eier_in_Eierkarton_6.jpg/960px-Eier_in_Eierkarton_6.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eier in Eierkarton 6.jpg",
  "CC BY-SA 4.0",
  "IgorCalzone1",
  437
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/cf/Gallus_gallus_MWNH_1135.JPG/960px-Gallus_gallus_MWNH_1135.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Gallus gallus MWNH 1135.JPG",
  "CC BY-SA 3.0",
  "Klaus Rassinger und Gerhard Cammerer, Museum Wiesbaden",
  493
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c7/Hand_holding_a_fresh_egg_from_a_chicken_coop.jpg/960px-Hand_holding_a_fresh_egg_from_a_chicken_coop.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Hand holding a fresh egg from a chicken coop.jpg",
  "CC BY 2.0",
  "Shixart1985",
  496
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2f/Hand_holds_a_raw_white_egg_in_a_dimly_lit_kitchen.jpg/960px-Hand_holds_a_raw_white_egg_in_a_dimly_lit_kitchen.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Hand holds a raw white egg in a dimly lit kitchen.jpg",
  "CC BY 2.0",
  "Shixart1985",
  497
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/9e/Jajci_s_slovenskimi_oznakami.jpg/960px-Jajci_s_slovenskimi_oznakami.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Jajci s slovenskimi oznakami.jpg",
  "CC BY-SA 4.0",
  "Melaleuca alternifolia",
  510
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/8/8a/Koni-juj.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Koni-juj.jpg",
  "CC BY-SA 4.0",
  "Parikhit phukan",
  514
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/aa/Kwai_na_kaza.jpg/960px-Kwai_na_kaza.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Kwai na kaza.jpg",
  "CC0",
  "Salaha mahmood",
  518
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6c/Kwan_kaza.jpg/960px-Kwan_kaza.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Kwan kaza.jpg",
  "CC0",
  "Umar A Muhammad",
  519
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4e/Meat_serves.jpg/960px-Meat_serves.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Meat serves.jpg",
  "CC BY-SA 3.0",
  "A8younan",
  529
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Multi-Colored_Eggs_%288065133518%29.jpg/960px-Multi-Colored_Eggs_%288065133518%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Multi-Colored Eggs (8065133518).jpg",
  "CC BY 2.0",
  "Larry Lamsa",
  533
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ea/Ous_falsos.jpg/960px-Ous_falsos.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ous falsos.jpg",
  "CC BY-SA 4.0",
  "GallRC",
  539
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/f/fd/Pigmentation_of_eggs._Incomplete_pigmentation_in_the_color_of_a_chicken_egg_shell._81.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Pigmentation of eggs. Incomplete pigmentation in the color of a chicken egg shell. 81.jpg",
  "CC BY-SA 4.0",
  "ВоєводаЯ",
  546
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/1/17/Pigmentation_of_eggs._Incomplete_pigmentation_in_the_color_of_a_chicken_egg_shell._83.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Pigmentation of eggs. Incomplete pigmentation in the color of a chicken egg shell. 83.jpg",
  "CC BY-SA 4.0",
  "ВоєводаЯ",
  548
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7c/Ramen_%40_Ichiran_%40_Shinjuku_%2815248167521%29.jpg/960px-Ramen_%40_Ichiran_%40_Shinjuku_%2815248167521%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ramen @ Ichiran @ Shinjuku (15248167521).jpg",
  "CC BY 2.0",
  "Guilhem Vellut from Annecy, France",
  556
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/eb/Silverudds_bl%C3%A5_%C3%A4gg.jpg/960px-Silverudds_bl%C3%A5_%C3%A4gg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Silverudds blå ägg.jpg",
  "CC BY-SA 4.0",
  "Ceciliaw79",
  562
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f1/Three_eggs_captured_in_May_2022.jpg/960px-Three_eggs_captured_in_May_2022.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Three eggs captured in May 2022.jpg",
  "CC BY-SA 4.0",
  "Billjones94",
  569
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/0e/Two_cups.jpg/960px-Two_cups.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Two cups.jpg",
  "CC BY 2.0",
  "darwin Bell from San Francisco, USA",
  574
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b0/%E1%80%80%E1%80%BC%E1%80%80%E1%80%BA%E1%80%A5.jpg/960px-%E1%80%80%E1%80%BC%E1%80%80%E1%80%BA%E1%80%A5.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:ကြက်ဥ.jpg",
  "CC BY-SA 4.0",
  "SarKaLay စာကလေး",
  595
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/3f/%E9%B8%A1%E8%9B%8B%EF%BC%88%E7%A9%BA%EF%BC%89_-_panoramio.jpg/960px-%E9%B8%A1%E8%9B%8B%EF%BC%88%E7%A9%BA%EF%BC%89_-_panoramio.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:鸡蛋（空） - panoramio.jpg",
  "CC BY-SA 3.0",
  "zhanyoun",
  600
 ]
]
CT_DIR = Path('/content/commons_train')
CT_DIR.mkdir(exist_ok=True)
for i, (url, *_resto) in enumerate(COMMONS_TRAIN):
    bajar(url, CT_DIR / f'{i:03d}.jpg')

etiquetas = (etiquetar(PUB_SRC / 'non defective', 1) + etiquetar(PUB_SRC / 'defective', 0)
             + etiquetar(CT_DIR, 1, origen='wtr'))
if PUB_DIR.exists():
    shutil.rmtree(PUB_DIR)
conteo = Counter()


def guardar_yolo(e, split, nombre):
    for sub in ('images', 'labels'):
        (PUB_DIR / split / sub).mkdir(parents=True, exist_ok=True)
    shutil.copy2(e['file'], PUB_DIR / split / 'images' / f'{nombre}{e["file"].suffix.lower()}')
    W, H = e['W'], e['H']
    (PUB_DIR / split / 'labels' / f'{nombre}.txt').write_text('\n'.join(
        f'{e["clase"]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}'
        for x1, y1, x2, y2 in e['cajas']) + '\n')


for origen, clase in [('pub', 1), ('pub', 0), ('wtr', 1)]:
    aceptadas = [e for e in etiquetas if e['clase'] == clase and e['ok'] and e['origen'] == origen]
    n = len(aceptadas)
    for i, e in enumerate(aceptadas):
        # dataset público: bloques consecutivos 70/15/15; fotos de Commons de entrenamiento: todas a train
        split = 'train' if (origen == 'wtr' or i < 0.70 * n) else 'valid' if i < 0.85 * n else 'test'
        e['split'] = split
        copias = 3 if (split == 'train' and clase == 1) else 1  # los sanos reales pesan más
        for k in range(copias):
            guardar_yolo(e, split, f'{origen}_{CLASSES[clase]}_{e["file"].stem}' + (f'_r{k}' if k else ''))
        conteo[(origen, CLASSES[clase], split)] += 1
pub_total = Counter(f"{e['origen']} {CLASSES[e['clase']]}" for e in etiquetas)
pub_ok = Counter(f"{e['origen']} {CLASSES[e['clase']]}" for e in etiquetas if e['ok'])
print('Fotos aceptadas / totales:', {c: f'{pub_ok[c]}/{pub_total[c]}' for c in sorted(pub_total)})
print('Por split (sin contar las repeticiones):', dict(sorted(conteo.items())))
assert pub_ok['pub Intact'] >= 60 and pub_ok['pub Crack'] >= 60, 'Muy pocas fotos aceptadas: revisa CONF_OK / CONF_DUDA.'

## 5. Auditoría de las cajas automáticas
Hojas con todas las fotos aceptadas y sus cajas (verde = sano, rojo = rajado). Se guardan en `runs/v3_auditoria` para revisarlas con calma.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

AUD_DIR = RUNS_DIR / 'v3_auditoria'
AUD_DIR.mkdir(parents=True, exist_ok=True)


def hoja(filas, destino, T=200, cols=8):
    rows = (len(filas) + cols - 1) // cols
    lienzo = np.full((rows * T, cols * T, 3), 30, np.uint8)
    for i, e in enumerate(filas):
        img = cv2.imread(str(e['file']))
        k = T / max(img.shape[:2])
        img = cv2.resize(img, (max(1, int(img.shape[1] * k)), max(1, int(img.shape[0] * k))))
        color = (0, 200, 0) if e['clase'] == 1 else (0, 0, 255)
        for x1, y1, x2, y2 in e['cajas']:
            cv2.rectangle(img, (int(x1 * k), int(y1 * k)), (int(x2 * k), int(y2 * k)), color, 2)
        cv2.putText(img, e['split'][:2], (3, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
        y, x = (i // cols) * T, (i % cols) * T
        lienzo[y:y + img.shape[0], x:x + img.shape[1]] = img
    cv2.imwrite(str(destino), lienzo, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return lienzo


for clase in (1, 0):
    acept = [e for e in etiquetas if e['clase'] == clase and e['ok']]
    for j in range(0, len(acept), 64):
        hoja(acept[j:j + 64], AUD_DIR / f'{CLASSES[clase]}_{j // 64:02d}.jpg')
json.dump([{'archivo': e['file'].name, 'clase': CLASSES[e['clase']], 'split': e.get('split'), 'aceptada': e['ok'],
            'cajas': [[round(v, 1) for v in b] for b in e['cajas']]} for e in etiquetas],
          open(AUD_DIR / 'etiquetas.json', 'w'), indent=0)

ejemplo = hoja([e for e in etiquetas if e['ok']][::12][:32], Path('/content/muestra.jpg'))
plt.figure(figsize=(18, 9)); plt.imshow(ejemplo[..., ::-1]); plt.axis('off'); plt.show()
print('Hojas guardadas en', AUD_DIR)

## 6. Dataset v3 y entrenamiento
`train` = dataset de `v2` + fotos reales; `valid` = valid de `v2` + fotos reales de valid (así `best.pt` se elige también por las fotos reales). Parte de `v2/best.pt`, mismos aumentos que `v2`, 30 épocas máx. con `patience=8`. Si `runs/v3` ya terminó, se salta; si quedó a medias, se reanuda.

In [ ]:
V3_YAML = V2_DIR / 'data_v3.yaml'
with open(V3_YAML, 'w') as f:
    yaml.safe_dump({'path': str(V2_DIR),
                    'train': [str(V2_DIR / 'train/images'), str(PUB_DIR / 'train/images')],
                    'val': [str(V2_DIR / 'valid/images'), str(PUB_DIR / 'valid/images')],
                    'test': str(DATA_DIR / 'test/images'), 'nc': 2, 'names': CLASSES}, f, sort_keys=False)

run_dir = RUNS_DIR / NEW_RUN
if (run_dir / 'results.png').exists():  # ultralytics solo crea results.png al terminar
    print(f'{NEW_RUN} ya está entrenado en {run_dir}: no se vuelve a entrenar.')
elif (run_dir / 'weights/last.pt').exists():
    print(f'{NEW_RUN} quedó a medias: se reanuda.')
    YOLO(str(run_dir / 'weights/last.pt')).train(resume=True)
else:
    assert not run_dir.exists(), f'{run_dir} existe pero no tiene pesos: revísalo (no se sobrescribe ningún run).'
    YOLO(str(BASE_PT)).train(
        data=str(V3_YAML),
        imgsz=640, epochs=30, patience=8, batch=-1, cache='disk',
        project=str(RUNS_DIR), name=NEW_RUN,
        hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
        degrees=15, translate=0.15, scale=0.5,
        fliplr=0.5, flipud=0.2,
        mosaic=1.0, mixup=0.1,
    )

## 7. Comparar v2 y v3 y elegir
- Métricas por clase en el **test original** (el mismo de `v1` y `v2`).
- Acierto por imagen y por grupo: test original, sintéticas de test y **fotos reales públicas de test** (grupo `real:`).
- **Wikimedia Commons**: fotos de otra fuente, etiquetadas a mano, que no se usan para entrenar ni para elegir. Acierto por foto: rajado si hay algún huevo `Crack`, sano si todos son `Intact`.

Se elige `v3` si mejora en las fotos reales sin empeorar el test original (mAP50-95 no baja más de 0.02 y ningún grupo `orig:` pierde más de 3 puntos).

In [ ]:
from ultralytics import YOLO

CONF = 0.5

def grupo_de(stem, clase_real):
    for t, nombre in [('_A_', 'A: Intact fuera del montaje'), ('_B_', 'B: Crack en el montaje'), ('_P_', 'P: fondo procedural')]:
        if stem.startswith('syn_') and t in stem:
            return nombre
    fuente = 'montaje' if stem.startswith('ec_egg') else 'otras fuentes'
    return f'orig: {clase_real} {fuente}'

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0

def leer_gt(labels_dir, stem):
    out = []
    for line in (labels_dir / f'{stem}.txt').read_text().splitlines():
        p = line.split()
        if len(p) == 5:
            c, xc, yc, w, h = int(p[0]), *map(float, p[1:])
            out.append((c, [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]))
    return out

def evaluar_imagenes(model, split_dir, conf=CONF):
    """Una fila por imagen: correcta si cada huevo tiene una predicción de su clase (IoU>=0.5) y no sobra nada."""
    filas = []
    # la carpeta (no una lista) para que prediga en lotes de 16 y no todas las imágenes a la vez en la GPU
    for r in model.predict(str(split_dir / 'images'), conf=conf, imgsz=640, batch=16, stream=True, verbose=False):
        path = Path(r.path)
        gt = leer_gt(split_dir / 'labels', path.stem)
        preds = list(zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(), r.boxes.xyxyn.tolist()))
        errores, usadas = [], set()
        for c, gbox in gt:
            solapan = [j for j, (_, _, b) in enumerate(preds) if iou(gbox, b) >= 0.5]
            usadas.update(solapan)
            clases = {preds[j][0] for j in solapan}
            if not solapan:
                errores.append(f'{CLASSES[c]} no detectado')
            elif c not in clases:
                errores.append(f'{CLASSES[c]} -> {CLASSES[1 - c]}')
            elif len(clases) > 1:
                errores.append('doble clase')
        errores += [f'sobra {CLASSES[pc]}' for j, (pc, _, _) in enumerate(preds) if j not in usadas]
        clase_real = CLASSES[gt[0][0]] if gt else 'fondo'
        filas.append({'img': path, 'grupo': grupo_de(path.stem, clase_real), 'correcto': not errores,
                      'errores': errores, 'gt': gt, 'preds': preds})
    return pd.DataFrame(filas)

def tabla_grupos(df):
    return df.groupby('grupo')['correcto'].agg(acierto='mean', correctas='sum', imagenes='count').round(3)



def grupo_de(stem, clase_real):
    if stem.startswith('pub_'):
        return f'real: {clase_real} fotos públicas'
    for t, nombre in [('_A_', 'A: Intact fuera del montaje'), ('_B_', 'B: Crack en el montaje'), ('_P_', 'P: fondo procedural')]:
        if stem.startswith('syn_') and t in stem:
            return nombre
    fuente = 'montaje' if stem.startswith('ec_egg') else 'otras fuentes'
    return f'orig: {clase_real} {fuente}'


def metricas_clase(model, data, nombre):
    m = model.val(data=str(data), split='test', imgsz=640, batch=16, plots=False,
                  project=str(RUNS_DIR), name=f'v3cmp_{nombre}', verbose=False)
    filas = {}
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        filas[CLASSES[c]] = {'P': p, 'R': r, 'mAP50': ap50, 'mAP50-95': ap}
    return m, pd.DataFrame(filas).T


COMMONS = [
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a7/Cracked_%2885516059%29.jpeg/960px-Cracked_%2885516059%29.jpeg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Cracked (85516059).jpeg",
  "CC BY 3.0",
  "\nCyd Haselton",
  10
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/54/Day_017_-_Photo365_-_Eggs_%2816299675421%29.jpg/960px-Day_017_-_Photo365_-_Eggs_%2816299675421%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Day 017 - Photo365 - Eggs (16299675421).jpg",
  "CC BY-SA 2.0",
  "UnknownNet Photography",
  12
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7f/Which_came_first%2C_the_chicken_or_the_egg_-_Flickr_-_katerha.jpg/960px-Which_came_first%2C_the_chicken_or_the_egg_-_Flickr_-_katerha.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Which came first, the chicken or the egg - Flickr - katerha.jpg",
  "CC BY 2.0",
  "Kate Ter Haar from Cedarville, MI, USA",
  44
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/40/Brown_egg_hatching.jpg/960px-Brown_egg_hatching.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Brown egg hatching.jpg",
  "CC BY-SA 3.0",
  "Linsenhejhej",
  102
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/db/Huehnerei_mit_Sprung_%28fcm%29.jpg/960px-Huehnerei_mit_Sprung_%28fcm%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Huehnerei mit Sprung (fcm).jpg",
  "CC BY-SA 3.0",
  "Frank C. Müller",
  173
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5d/Cracked_egg_%2830298255024%29.jpg/960px-Cracked_egg_%2830298255024%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Cracked egg (30298255024).jpg",
  "CC BY 2.0",
  "Carlos Ebert from São Paulo, BrazilGRU",
  608
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4a/Bowl_of_Eggs_%28Unsplash%29.jpg/960px-Bowl_of_Eggs_%28Unsplash%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Crack",
  "File:Bowl of Eggs (Unsplash).jpg",
  "CC0",
  "Cory Seward cseward15",
  674
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f8/08726jfFilipino_foods_fruits_Bulacan_landmarksfvf_46.jpg/960px-08726jfFilipino_foods_fruits_Bulacan_landmarksfvf_46.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:08726jfFilipino foods fruits Bulacan landmarksfvf 46.jpg",
  "CC0",
  "Judgefloro",
  55
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f4/08800jfFilipino_foods_fruits_Bulacan_landmarksfvf_17.jpg/960px-08800jfFilipino_foods_fruits_Bulacan_landmarksfvf_17.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:08800jfFilipino foods fruits Bulacan landmarksfvf 17.jpg",
  "CC0",
  "Judgefloro",
  56
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f0/08800jfFilipino_foods_fruits_Bulacan_landmarksfvf_18.jpg/960px-08800jfFilipino_foods_fruits_Bulacan_landmarksfvf_18.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:08800jfFilipino foods fruits Bulacan landmarksfvf 18.jpg",
  "CC0",
  "Judgefloro",
  57
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/23/08800jfFilipino_foods_fruits_Bulacan_landmarksfvf_21.jpg/960px-08800jfFilipino_foods_fruits_Bulacan_landmarksfvf_21.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:08800jfFilipino foods fruits Bulacan landmarksfvf 21.jpg",
  "CC0",
  "Judgefloro",
  58
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e8/09042jfCuisine_Foods_Philippines_Baliuag_Bulacanfvf_03.jpg/960px-09042jfCuisine_Foods_Philippines_Baliuag_Bulacanfvf_03.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:09042jfCuisine Foods Philippines Baliuag Bulacanfvf 03.jpg",
  "CC0",
  "Judgefloro",
  59
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/85/09042jfCuisine_Foods_Philippines_Baliuag_Bulacanfvf_05.jpg/960px-09042jfCuisine_Foods_Philippines_Baliuag_Bulacanfvf_05.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:09042jfCuisine Foods Philippines Baliuag Bulacanfvf 05.jpg",
  "CC0",
  "Judgefloro",
  60
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/15/2_eggs.JPG/960px-2_eggs.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 eggs.JPG",
  "CC BY-SA 4.0",
  "Hannahdownes",
  61
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/58/2_primeros_huevos_isa_brown_%286%29.JPG/960px-2_primeros_huevos_isa_brown_%286%29.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 primeros huevos isa brown (6).JPG",
  "CC BY-SA 3.0",
  "Lanntaron",
  62
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c3/2_primeros_huevos_isa_brown.JPG/960px-2_primeros_huevos_isa_brown.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2 primeros huevos isa brown.JPG",
  "CC BY-SA 3.0",
  "Lanntaron",
  63
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/da/8451Poblacion%2C_Baliuag%2C_Bulacan_12.jpg/960px-8451Poblacion%2C_Baliuag%2C_Bulacan_12.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:8451Poblacion, Baliuag, Bulacan 12.jpg",
  "CC0",
  "Judgefloro",
  71
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/01/8451Poblacion%2C_Baliuag%2C_Bulacan_15.jpg/960px-8451Poblacion%2C_Baliuag%2C_Bulacan_15.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:8451Poblacion, Baliuag, Bulacan 15.jpg",
  "CC0",
  "Judgefloro",
  74
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2b/8451Poblacion%2C_Baliuag%2C_Bulacan_19.jpg/960px-8451Poblacion%2C_Baliuag%2C_Bulacan_19.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:8451Poblacion, Baliuag, Bulacan 19.jpg",
  "CC0",
  "Judgefloro",
  78
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/db/AraucanaEgg_vs_Brown_White.jpg/960px-AraucanaEgg_vs_Brown_White.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:AraucanaEgg vs Brown White.jpg",
  "Public domain",
  "Gmoose1",
  86
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/aa/AT-Eierkennzeichnung.jpg/960px-AT-Eierkennzeichnung.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:AT-Eierkennzeichnung.jpg",
  "CC BY-SA 3.0",
  "Thomas R. Schwarz",
  87
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1b/AZ_provincia.jpg/960px-AZ_provincia.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:AZ provincia.jpg",
  "CC BY-SA 4.0",
  "Marica Massaro",
  88
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/50/Bain_d%27eau_chaude_%283526896150%29.jpg/960px-Bain_d%27eau_chaude_%283526896150%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Bain d'eau chaude (3526896150).jpg",
  "CC BY 2.0",
  "The Marmot",
  91
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5e/Berlin_frische_Eier_aus_privater_Haltung.jpg/960px-Berlin_frische_Eier_aus_privater_Haltung.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Berlin frische Eier aus privater Haltung.jpg",
  "CC BY 4.0",
  "Lukas Beck",
  92
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/0/02/Blue_Araucana_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Blue Araucana egg.jpg",
  "CC BY-SA 2.0",
  "Julian Berry from Mid Sussex, UK",
  95
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/17/Bring_on_your_most_eggscruciating_puns%21_%286925079769%29.jpg/960px-Bring_on_your_most_eggscruciating_puns%21_%286925079769%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Bring on your most eggscruciating puns! (6925079769).jpg",
  "CC BY-SA 2.0",
  "theilr",
  97
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/8/81/Brown_chicken_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Brown chicken egg.jpg",
  "Public domain",
  "Garitzko",
  99
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e6/Brown_chicken_eggs_%281%29.jpg/960px-Brown_chicken_eggs_%281%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Brown chicken eggs (1).jpg",
  "CC BY 2.0",
  "Justus Blümer from Deutschland",
  100
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/01/Brown_chicken_eggs.jpg/960px-Brown_chicken_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Brown chicken eggs.jpg",
  "CC BY-SA 3.0",
  "Piasoft",
  101
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/86/Brown_egg.jpg/960px-Brown_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Brown egg.jpg",
  "CC BY-SA 1.0",
  "George Shuklin",
  104
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b1/Carton_of_eggs.jpg/960px-Carton_of_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Carton of eggs.jpg",
  "CC BY 3.0",
  "Gisela Francisco",
  110
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5e/Chicken_egg_2009-06-04.jpg/960px-Chicken_egg_2009-06-04.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken egg 2009-06-04.jpg",
  "CC BY-SA 3.0",
  "Sun Ladder",
  111
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/78/Chicken_egg.jpg/960px-Chicken_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken egg.jpg",
  "CC BY-SA 3.0",
  "Javier8aflores",
  112
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/99/Chicken_eggs_21.jpg/960px-Chicken_eggs_21.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken eggs 21.jpg",
  "CC BY-SA 4.0",
  "BogTar201213",
  113
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/b/be/Chicken-Eggs_339411-480x360_%284900327148%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Chicken-Eggs 339411-480x360 (4900327148).jpg",
  "CC BY 2.0",
  "Emilian Robert Vicol from Com. Balanesti, Romania",
  115
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/c/c6/Chicken-Eggs_368816-480x360_%284900327050%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Chicken-Eggs 368816-480x360 (4900327050).jpg",
  "CC BY 2.0",
  "Emilian Robert Vicol from Com. Balanesti, Romania",
  116
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/3b/Deux_oeufs_bruns.jpg/960px-Deux_oeufs_bruns.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Deux oeufs bruns.jpg",
  "CC BY-SA 4.0",
  "Jeangagnon",
  117
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f0/Different_colored_chicken_eggs.jpg/960px-Different_colored_chicken_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Different colored chicken eggs.jpg",
  "CC BY-SA 4.0",
  "Charlotte Corbisier",
  119
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/00/Egg_%28394351849%29.jpg/960px-Egg_%28394351849%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg (394351849).jpg",
  "CC BY 2.0",
  "liz west from Boxborough, MA, USA",
  124
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ee/Egg_colours.jpg/960px-Egg_colours.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg colours.jpg",
  "CC BY-SA 3.0",
  "Timothy Titus",
  125
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/98/Egg_food.jpg/960px-Egg_food.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg food.jpg",
  "Public domain",
  "Paolo Neo",
  126
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a3/Egg_on_Table.jpg/960px-Egg_on_Table.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg on Table.jpg",
  "CC BY-SA 2.0",
  "Eric Petruno",
  127
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/67/Egg_on_white_background.jpg/960px-Egg_on_white_background.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg on white background.jpg",
  "Public domain",
  "Paolo Neo",
  128
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/23/Egg_perched_uplon_a_wooden_fence_2005.jpg/960px-Egg_perched_uplon_a_wooden_fence_2005.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg perched uplon a wooden fence 2005.jpg",
  "CC BY 2.0",
  "Dave&amp;Lynne Slater from Cumbria, United kingdom",
  129
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/Egg_spiral_egg_cup.jpg/960px-Egg_spiral_egg_cup.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg spiral egg cup.jpg",
  "CC BY 2.5",
  "Marie-Lan Nguyen\n",
  130
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/7b/Egg_upright.jpg/960px-Egg_upright.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg upright.jpg",
  "Public domain",
  "Paolo Neo",
  132
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/2/2e/Egg_W%26A.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Egg W&A.jpg",
  "Copyrighted free use-link",
  "The original uploader was Wajiduddaim at English Wikipedia.",
  133
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4b/Eggs_%285219850239%29.jpg/960px-Eggs_%285219850239%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs (5219850239).jpg",
  "CC BY 2.0",
  "Jennifer C.",
  136
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/61/Eggs_%287177637421%29.jpg/960px-Eggs_%287177637421%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs (7177637421).jpg",
  "CC BY 2.0",
  "James Bowe",
  139
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f0/Eggs_%288501854880%29.jpg/960px-Eggs_%288501854880%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs (8501854880).jpg",
  "CC BY 2.0",
  "liz west from Boxborough, MA, USA",
  141
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/04/Eggs_-_Home_Grown_%28or_laid%29_%287147848749%29.jpg/960px-Eggs_-_Home_Grown_%28or_laid%29_%287147848749%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs - Home Grown (or laid) (7147848749).jpg",
  "CC BY 2.0",
  "craigles75",
  142
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/31/Eggs_3145g.jpg/960px-Eggs_3145g.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs 3145g.jpg",
  "CC BY-SA 4.0",
  "StaraBlazkova",
  144
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/a4/Eggs_99_365_%283424917922%29.jpg/960px-Eggs_99_365_%283424917922%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs 99 365 (3424917922).jpg",
  "CC BY 2.0",
  "julie from Idaho, united states",
  145
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b6/Eggs_green_brown_on_end.jpg/960px-Eggs_green_brown_on_end.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eggs green brown on end.jpg",
  "CC BY-SA 3.0",
  "Hustvedt",
  146
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d5/Ei_kennzeichnung.jpg/960px-Ei_kennzeichnung.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Ei kennzeichnung.jpg",
  "CC BY-SA 3.0",
  "No machine-readable author provided. Abdull assumed (based on copyright claims).",
  149
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/fb/Eier_%2812163693795%29.jpg/960px-Eier_%2812163693795%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eier (12163693795).jpg",
  "CC BY 2.0",
  "Tim Reckmann from Hamm, Deutschland",
  151
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/df/Eier_%286045563994%29.jpg/960px-Eier_%286045563994%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eier (6045563994).jpg",
  "CC BY 2.0",
  "Justus Blümer from Deutschland",
  152
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/40/Eieren_%287937424642%29.jpg/960px-Eieren_%287937424642%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Eieren (7937424642).jpg",
  "CC BY 2.0",
  "Serita.V",
  157
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/c/c4/Eierkennzeichnung_BMK.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Eierkennzeichnung BMK.jpg",
  "CC BY-SA 3.0",
  "User:BMK",
  158
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/fe/File_by_Alexander_Baranov_-_%2819411031842%29.jpg/960px-File_by_Alexander_Baranov_-_%2819411031842%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:File by Alexander Baranov - (19411031842).jpg",
  "CC BY 2.0",
  "Alexander Baranov from Montpellier, France",
  159
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8f/First_egg_%287680068562%29.jpg/960px-First_egg_%287680068562%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:First egg (7680068562).jpg",
  "CC BY 2.0",
  "normanack",
  160
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f8/Flickr_-_cyclonebill_-_%C3%86g_%281%29.jpg/960px-Flickr_-_cyclonebill_-_%C3%86g_%281%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Flickr - cyclonebill - Æg (1).jpg",
  "CC BY-SA 2.0",
  "cyclonebill",
  161
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/8/80/Fran%C3%A7oise_Foliot_-_Un_oeuf_de_poule.jpg/960px-Fran%C3%A7oise_Foliot_-_Un_oeuf_de_poule.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Françoise Foliot - Un oeuf de poule.jpg",
  "CC BY-SA 4.0",
  "Françoise Foliot",
  163
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/75/FreeRangeEgg.jpg/960px-FreeRangeEgg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:FreeRangeEgg.jpg",
  "CC BY-SA 3.0",
  "",
  165
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4a/Fresh_Eggs_%28Unsplash_fBturNUtd8%29.jpg/960px-Fresh_Eggs_%28Unsplash_fBturNUtd8%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Fresh Eggs (Unsplash fBturNUtd8).jpg",
  "CC0",
  "Brina Blum brina_blum",
  166
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/56/Giriraja_eggs.jpg/960px-Giriraja_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Giriraja eggs.jpg",
  "CC BY-SA 3.0",
  "Netha Hussain",
  167
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/35/Hikari%2C_Eggs_rich_in_iodine.jpg/960px-Hikari%2C_Eggs_rich_in_iodine.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Hikari, Eggs rich in iodine.jpg",
  "CC BY-SA 4.0",
  "Miyuki Meinaka",
  169
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6f/Huevos_de_gallina_de_diferente_tama%C3%B1o.JPG/960px-Huevos_de_gallina_de_diferente_tama%C3%B1o.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Huevos de gallina de diferente tamaño.JPG",
  "CC BY-SA 4.0",
  "John PC",
  176
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/7/76/Just_lying_there_being_pretty_%284301382316%29.jpg/960px-Just_lying_there_being_pretty_%284301382316%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Just lying there being pretty (4301382316).jpg",
  "CC BY 2.0",
  "storebukkebruse",
  180
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/5/56/MonstereiXXXL.jpg/960px-MonstereiXXXL.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:MonstereiXXXL.jpg",
  "CC BY-SA 3.0",
  "-jkb-",
  186
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/3/39/Oeuf_de_poule.jpg/960px-Oeuf_de_poule.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Oeuf de poule.jpg",
  "CC BY-SA 4.0",
  "Jeangagnon",
  187
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b1/Oeufs_%2817056124402%29.jpg/960px-Oeufs_%2817056124402%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Oeufs (17056124402).jpg",
  "CC BY-SA 2.0",
  "Lapichon",
  188
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6f/Pastel_de_pollo_-_ingredientes_%282%29.jpg/960px-Pastel_de_pollo_-_ingredientes_%282%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Pastel de pollo - ingredientes (2).jpg",
  "CC BY 3.0",
  "Juan Gonzalo Angel Retrepo",
  191
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/e/e8/Protein-rich_Foods.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Protein-rich Foods.jpg",
  "CC BY-SA 4.0",
  "Smastronardo",
  193
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/62/Respeggt_eggs_2.jpg/960px-Respeggt_eggs_2.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Respeggt eggs 2.jpg",
  "CC BY-SA 4.0",
  "Nederlandse Leeuw",
  196
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/02/Respeggt_eggs_3.jpg/960px-Respeggt_eggs_3.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Respeggt eggs 3.jpg",
  "CC BY-SA 4.0",
  "Nederlandse Leeuw",
  197
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/26/Six_eggs_views_from_the_top_on_a_white_background.jpg/960px-Six_eggs_views_from_the_top_on_a_white_background.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Six eggs views from the top on a white background.jpg",
  "CC BY-SA 4.0",
  "TudorTulok",
  199
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4f/Spanish_eggs.jpg/960px-Spanish_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Spanish eggs.jpg",
  "Public domain",
  "Lameiro at Galician Wikipedia",
  200
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/df/Spotted_Brown_Chicken_Eggs_%285719609727%29.jpg/960px-Spotted_Brown_Chicken_Eggs_%285719609727%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Spotted Brown Chicken Eggs (5719609727).jpg",
  "CC BY 2.0",
  "spiralmushroom",
  202
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/2/20/Standard_egg_sizes.jpg/960px-Standard_egg_sizes.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Standard egg sizes.jpg",
  "Public domain",
  "Johan6865",
  203
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/09/Three_chicken_eggs.jpg/960px-Three_chicken_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Three chicken eggs.jpg",
  "CC0",
  "www.Pixel.la Free Stock Photos",
  205
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/cf/Tiny_treasure%2C_first_egg_%287680059460%29.jpg/960px-Tiny_treasure%2C_first_egg_%287680059460%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Tiny treasure, first egg (7680059460).jpg",
  "CC BY 2.0",
  "normanack",
  206
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d1/Toj%C3%A1s.jpg/960px-Toj%C3%A1s.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Tojás.jpg",
  "CC BY-SA 3.0",
  "Safranyl",
  207
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e3/%D0%90%D0%BA%D1%83%D0%BB%D0%B8%D0%BD%D0%BE%D0%B2%D0%BA%D0%B0_26.jpg/960px-%D0%90%D0%BA%D1%83%D0%BB%D0%B8%D0%BD%D0%BE%D0%B2%D0%BA%D0%B0_26.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Акулиновка 26.jpg",
  "CC BY-SA 4.0",
  "Ситников Алексей Александрович",
  211
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ea/001_2022_04_09_Ei.jpg/960px-001_2022_04_09_Ei.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:001 2022 04 09 Ei.jpg",
  "CC BY-SA 4.0",
  "Friedrich Haag",
  222
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/92/2020-04-27_12_12_45_A_single_Medium_Grade_A_Chicken_Egg_from_Hillandale_Farms_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg/960px-2020-04-27_12_12_45_A_single_Medium_Grade_A_Chicken_Egg_from_Hillandale_Farms_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2020-04-27 12 12 45 A single Medium Grade A Chicken Egg from Hillandale Farms in the Franklin Farm section of Oak Hill, Fairfax County, Virginia.jpg",
  "CC BY-SA 4.0",
  "Famartin",
  231
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/b/bb/2020-04-27_12_16_12_A_single_Medium_Grade_A_Chicken_Egg_from_Hillandale_Farms_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg/960px-2020-04-27_12_16_12_A_single_Medium_Grade_A_Chicken_Egg_from_Hillandale_Farms_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2020-04-27 12 16 12 A single Medium Grade A Chicken Egg from Hillandale Farms in the Franklin Farm section of Oak Hill, Fairfax County, Virginia.jpg",
  "CC BY-SA 4.0",
  "Famartin",
  232
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/66/2020-05-05_18_22_46_Six_eggs_in_an_open_carton_of_a_dozen_Large_Grade_A_Chicken_Eggs_from_Egg-land%27s_Best_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg/960px-thumbnail.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2020-05-05 18 22 46 Six eggs in an open carton of a dozen Large Grade A Chicken Eggs from Egg-land's Best in the Franklin Farm section of Oak Hill, Fairfax County, Virginia.jpg",
  "CC BY-SA 4.0",
  "Famartin",
  234
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/41/2020-05-05_19_30_03_A_single_Large_Grade_A_Chicken_Egg_from_Egg-land%27s_Best_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg/960px-2020-05-05_19_30_03_A_single_Large_Grade_A_Chicken_Egg_from_Egg-land%27s_Best_in_the_Franklin_Farm_section_of_Oak_Hill%2C_Fairfax_County%2C_Virginia.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2020-05-05 19 30 03 A single Large Grade A Chicken Egg from Egg-land's Best in the Franklin Farm section of Oak Hill, Fairfax County, Virginia.jpg",
  "CC BY-SA 4.0",
  "Famartin",
  240
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/60/2eggs.jpg/960px-2eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:2eggs.jpg",
  "CC BY 3.0",
  "ZabMilenko",
  241
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/66/3_Egg_white.JPG/960px-3_Egg_white.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:3 Egg white.JPG",
  "CC BY-SA 4.0",
  "MOs810",
  242
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/10/3eggs.jpg/960px-3eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:3eggs.jpg",
  "CC BY 3.0",
  "ZabMilenko",
  243
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/b/bc/Banana_egg_pancakes_-_ingredients_%2816842815337%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Banana egg pancakes - ingredients (16842815337).jpg",
  "CC BY-SA 2.0",
  "sunny mama",
  244
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ee/Blau_Eier_05.jpg/960px-Blau_Eier_05.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Blau Eier 05.jpg",
  "CC BY-SA 3.0",
  "Penarc",
  245
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/91/Chicken_Egg_-_%E0%B4%95%E0%B5%8B%E0%B4%B4%E0%B4%BF%E0%B4%AE%E0%B5%81%E0%B4%9F%E0%B5%8D%E0%B4%9F_03.JPG/960px-Chicken_Egg_-_%E0%B4%95%E0%B5%8B%E0%B4%B4%E0%B4%BF%E0%B4%AE%E0%B5%81%E0%B4%9F%E0%B5%8D%E0%B4%9F_03.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken Egg - കോഴിമുട്ട 03.JPG",
  "CC BY-SA 3.0",
  "കാക്കര",
  248
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/6/64/Chicken_eggs_20101113.jpg/960px-Chicken_eggs_20101113.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Chicken eggs 20101113.jpg",
  "Public domain",
  "Batholith (talk)",
  251
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/0/0e/Chicken_eggs.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Chicken eggs.jpg",
  "CC BY-SA 3.0",
  "Fir0002",
  252
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/a/a5/CodazziToyotaMorning1.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:CodazziToyotaMorning1.jpg",
  "CC BY-SA 3.0",
  "",
  253
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/19/Egg_%284245577202%29.jpg/960px-Egg_%284245577202%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Egg (4245577202).jpg",
  "CC BY 2.0",
  "Gabriel Rocha from Rio de Janeiro, Brazil",
  255
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/First_three_egg_day_in_eons_%288529932306%29.jpg/960px-First_three_egg_day_in_eons_%288529932306%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:First three egg day in eons (8529932306).jpg",
  "CC BY 2.0",
  "Steven-L-Johnson",
  258
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/0/02/Kipppunkt_Ei.jpg/960px-Kipppunkt_Ei.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Kipppunkt Ei.jpg",
  "CC BY 3.0",
  "Jovel",
  263
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c8/Norfolk_Grey_egg_%2814827770651%29.jpg/960px-Norfolk_Grey_egg_%2814827770651%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Norfolk Grey egg (14827770651).jpg",
  "CC BY-SA 2.0",
  "Kat from UK",
  267
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d3/Pair_of_organic_green_eggs.JPG/960px-Pair_of_organic_green_eggs.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Pair of organic green eggs.JPG",
  "CC BY 1.0",
  "VanTucky",
  269
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f5/Raw_Egg.JPG/960px-Raw_Egg.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Raw Egg.JPG",
  "CC BY-SA 4.0",
  "Kumard.ashish",
  270
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e9/Telur_ayam_kampung.JPG/960px-Telur_ayam_kampung.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Telur ayam kampung.JPG",
  "CC BY 3.0",
  "Sakurai Midori",
  271
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/1/1b/White_chicken_egg_square.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:White chicken egg square.jpg",
  "CC BY 2.0",
  "Ren West, square crop by uploader",
  276
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e5/White_chicken_egg.jpg/960px-White_chicken_egg.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:White chicken egg.jpg",
  "CC BY 2.0",
  "Ren West",
  277
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/46/%D0%AF%D0%B9%D1%86%D0%BE_%D1%81%D0%BE_%D1%81%D1%82%D1%80%D0%B0%D1%88%D0%BD%D0%BE%D0%B9_%D0%BC%D0%B0%D1%80%D0%BA%D0%B8%D1%80%D0%BE%D0%B2%D0%BA%D0%BE%D0%B9.jpg/960px-%D0%AF%D0%B9%D1%86%D0%BE_%D1%81%D0%BE_%D1%81%D1%82%D1%80%D0%B0%D1%88%D0%BD%D0%BE%D0%B9_%D0%BC%D0%B0%D1%80%D0%BA%D0%B8%D1%80%D0%BE%D0%B2%D0%BA%D0%BE%D0%B9.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Яйцо со страшной маркировкой.jpg",
  "CC0",
  "Retired electrician",
  281
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c8/%D8%AA%D9%88%D8%A3%D9%85_%D8%AD%D9%85%D8%A7%D9%85.JPG/960px-%D8%AA%D9%88%D8%A3%D9%85_%D8%AD%D9%85%D8%A7%D9%85.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:توأم حمام.JPG",
  "CC BY-SA 3.0",
  "Hind.qm",
  282
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4a/%E0%AE%95%E0%AF%8B%E0%AE%B4%E0%AE%BF_%E0%AE%AE%E0%AF%81%E0%AE%9F%E0%AF%8D%E0%AE%9F%E0%AF%88.JPG/960px-%E0%AE%95%E0%AF%8B%E0%AE%B4%E0%AE%BF_%E0%AE%AE%E0%AF%81%E0%AE%9F%E0%AF%8D%E0%AE%9F%E0%AF%88.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:கோழி முட்டை.JPG",
  "CC BY-SA 3.0",
  "AMALAN619",
  285
 ],
 [
  "https://upload.wikimedia.org/wikipedia/commons/9/9e/Senegal_egg_10s06.JPG?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
  "Intact",
  "File:Senegal egg 10s06.JPG",
  "CC BY-SA 3.0",
  "Snowmanradio at English Wikipedia\n\n(Original text: snowmanradio)",
  615
 ],
 [
  "https://thumb.wikimedia.org/wikipedia/commons/thumb/9/9a/Wood_duck_egg_2012.jpg/960px-Wood_duck_egg_2012.jpg?utm_source=commons.wikimedia.org&utm_campaign=imageinfo&utm_content=thumbnail",
  "Intact",
  "File:Wood duck egg 2012.jpg",
  "Public domain",
  "Marines.mil",
  749
 ]
]
COMMONS_DIR = Path('/content/commons')
COMMONS_DIR.mkdir(exist_ok=True)
COMMONS_OK = [bajar(url, COMMONS_DIR / f'{i:03d}.jpg') for i, (url, *_resto) in enumerate(COMMONS)]


def evaluar_commons(model, conf=CONF):
    filas = []
    for i, (url, clase, titulo, *_resto) in enumerate(COMMONS):
        if not COMMONS_OK[i]:
            continue
        r = model.predict(str(COMMONS_DIR / f'{i:03d}.jpg'), conf=conf, imgsz=640, verbose=False)[0]
        cls = r.boxes.cls.int().tolist()
        pred = 'Ninguno' if not cls else ('Crack' if 0 in cls else 'Intact')
        filas.append({'titulo': titulo, 'clase': clase, 'pred': pred, 'correcto': pred == clase})
    return pd.DataFrame(filas)


print(f'Commons: {sum(COMMONS_OK)} de {len(COMMONS)} fotos descargadas ({Counter(c for _, c, *_r in COMMONS)})')

In [ ]:
model_v2 = YOLO(str(BASE_PT))
model_v3 = YOLO(str(RUNS_DIR / NEW_RUN / 'weights/best.pt'))
comparacion, grupos, commons = {}, {}, {}
for nombre, mdl in [(BASE_RUN, model_v2), (NEW_RUN, model_v3)]:
    _, comparacion[nombre] = metricas_clase(mdl, DATA_V2, f'{nombre}_test_orig')
    df = pd.concat([evaluar_imagenes(mdl, DATA_DIR / 'test'), evaluar_imagenes(mdl, V2_DIR / 'test_synth'),
                    evaluar_imagenes(mdl, PUB_DIR / 'test')])
    grupos[nombre] = tabla_grupos(df)['acierto']
    commons[nombre] = evaluar_commons(mdl)
    if nombre == NEW_RUN:
        df_v3 = df
display(pd.concat(comparacion, axis=1).round(3))
grupos = pd.concat(grupos, axis=1)
display(grupos)
tabla_commons = pd.DataFrame({n: d.groupby('clase')['correcto'].mean() for n, d in commons.items()})
tabla_commons.loc['total'] = [d['correcto'].mean() for d in commons.values()]
print('Wikimedia Commons (test independiente, acierto por foto):')
display(tabla_commons.round(3))

orig = grupos.index.str.startswith('orig:')
reales = grupos.index.str.startswith('real:')
caida_grupos = float((grupos.loc[orig, BASE_RUN] - grupos.loc[orig, NEW_RUN]).max())
caida_map = float(comparacion[BASE_RUN]['mAP50-95'].mean() - comparacion[NEW_RUN]['mAP50-95'].mean())
mejora_reales = float(grupos.loc[reales, NEW_RUN].mean() - grupos.loc[reales, BASE_RUN].mean())
ELEGIDO = NEW_RUN if (caida_map < 0.02 and caida_grupos < 0.03 and mejora_reales > 0.05) else BASE_RUN
print(f'Test original: mAP50-95 {-caida_map:+.3f}, peor grupo orig {-caida_grupos:+.3f} | '
      f'fotos reales: {mejora_reales:+.3f} de acierto medio')
print(f'-> Modelo elegido: {ELEGIDO}')

# La comparación se guarda siempre, gane o no v3
comparacion_json = {
    'elegido': ELEGIDO,
    'criterio': {'caida_mAP50_95_test_orig': round(caida_map, 4), 'peor_caida_grupo_orig': round(caida_grupos, 4),
                 'mejora_fotos_reales_publicas': round(mejora_reales, 4)},
    'test_original_por_clase': {n: t.round(4).to_dict() for n, t in comparacion.items()},
    'acierto_por_grupo': grupos.round(4).to_dict(),
    'commons': {n: {'acierto': round(float(d['correcto'].mean()), 4), 'n': len(d),
                    'por_clase': d.groupby('clase')['correcto'].mean().round(4).to_dict(),
                    'detalle': d.to_dict(orient='records')} for n, d in commons.items()},
    'fotos_reales_aceptadas': dict(pub_ok), 'fotos_reales_totales': dict(pub_total),
}
(RUNS_DIR / 'v3_comparacion.json').write_text(json.dumps(comparacion_json, indent=1, ensure_ascii=False, default=str))
print('Comparación guardada en', RUNS_DIR / 'v3_comparacion.json')

## 8. Errores de v3 en fotos reales
Hasta 12 fotos reales públicas de test mal clasificadas. Caja blanca = etiqueta, roja = predicción.

In [ ]:
errores = df_v3[(~df_v3['correcto']) & df_v3['grupo'].str.startswith('real:')]
print(f'{len(errores)} fotos reales de test con error')
if len(errores):
    muestra = errores.sample(min(12, len(errores)), random_state=0)
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for ax, (_, fila) in zip(axes.flat, muestra.iterrows()):
        img = cv2.cvtColor(cv2.imread(str(fila['img'])), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        t = max(1, round(min(H, W) / 150))
        for _, b in fila['gt']:
            cv2.rectangle(img, (int(b[0] * W), int(b[1] * H)), (int(b[2] * W), int(b[3] * H)), (255, 255, 255), t)
        for pc, pconf, b in fila['preds']:
            cv2.rectangle(img, (int(b[0] * W), int(b[1] * H)), (int(b[2] * W), int(b[3] * H)), (255, 0, 0), t)
        ax.imshow(img)
        ax.set_title(f"{fila['grupo']}\n{', '.join(fila['errores'])}", fontsize=8, color='red')
    for ax in axes.flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 9. Exportar a TFLite (LiteRT) con entrada NHWC
Misma exportación que `v2` (celda 9 de `eggs_v2.ipynb`): entrada NHWC `[1, 640, 640, 3]`, salida `[1, 6, 8400]` con cajas normalizadas. INT8 se calibra con imágenes de valid de `v3`. Solo se ejecuta si se eligió `v3`.

In [ ]:
if ELEGIDO != NEW_RUN:
    print('v3 no mejoró a v2: no se exporta nada y se mantiene v2.')
else:
    import torch
    import ultralytics.utils.export.litert as litert_mod
    from ultralytics.utils.export.engine import _NormalizeCoords

    _torch2litert_original = litert_mod.torch2litert

    class _EntradaNHWC(torch.nn.Module):
        """Recibe [1, H, W, 3] (lo que entrega la cámara) y lo pasa al modelo como [1, 3, H, W]."""
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, x):
            return self.model(x.permute(0, 3, 1, 2))

    class _CalibracionNHWC:
        """Pasa las imágenes de calibración INT8 a NHWC para que coincidan con la nueva entrada."""
        def __init__(self, loader):
            self.loader = loader
        def __iter__(self):
            for batch in self.loader:
                batch = dict(batch)
                batch['img'] = batch['img'].permute(0, 2, 3, 1)
                yield batch

    def _torch2litert_nhwc(model, im, file, quantize, calibration_dataset, metadata, prefix):
        h, w = int(im.shape[2]), int(im.shape[3])
        model = _EntradaNHWC(_NormalizeCoords(model, h, w, 'detect', len((metadata or {}).get('names', {})), None))
        normalize_original = litert_mod._NormalizeCoords
        litert_mod._NormalizeCoords = lambda m, *a, **k: m  # ya normalizado arriba
        try:
            return _torch2litert_original(
                model, im.permute(0, 2, 3, 1).contiguous(), file, quantize,
                _CalibracionNHWC(calibration_dataset) if calibration_dataset is not None else None, metadata, prefix)
        finally:
            litert_mod._NormalizeCoords = normalize_original

    litert_mod.torch2litert = _torch2litert_nhwc

    EXPORT_DIR = EXPORTS_DIR / ELEGIDO
    TFLITE = {'fp32': EXPORT_DIR / f'eggs_{ELEGIDO}_fp32.tflite', 'int8': EXPORT_DIR / f'eggs_{ELEGIDO}_int8.tflite'}
    pt = EXPORT_DIR / f'eggs_{ELEGIDO}.pt'
    if all(f.exists() for f in [*TFLITE.values(), pt]):
        print(f'Ya exportado en {EXPORT_DIR}: se reutiliza.')
    else:
        assert not EXPORT_DIR.exists(), f'{EXPORT_DIR} existe pero está incompleto: revísalo antes de exportar de nuevo.'
        work = Path('/content/export') / ELEGIDO
        if work.exists():
            shutil.rmtree(work)
        work.mkdir(parents=True)
        pt_local = work / pt.name
        shutil.copy2(RUNS_DIR / ELEGIDO / 'weights/best.pt', pt_local)

        f_fp32 = Path(YOLO(str(pt_local)).export(format='litert', imgsz=640))
        f_int8 = Path(YOLO(str(pt_local)).export(format='litert', imgsz=640, quantize=8, data=str(V3_YAML), fraction=0.5))

        EXPORT_DIR.mkdir(parents=True)
        shutil.copy2(f_fp32, TFLITE['fp32'])
        shutil.copy2(f_int8, TFLITE['int8'])
        shutil.copy2(pt_local, pt)
    for k, f in TFLITE.items():
        print(f'{k}: {f}  ({f.stat().st_size / 1e6:.1f} MB)')

## 10. Verificar los .tflite y guardar el resumen
Compara `.pt`, FP32 e INT8 en test original + sintéticas, comprueba la decodificación de la app y guarda `resumen.json` junto a los `.tflite`.

In [ ]:
if ELEGIDO == NEW_RUN:
    CONF_APP = CONF
    DATA_TEST_ALL = V2_DIR / 'test_all.yaml'
    import json
    from ai_edge_litert.interpreter import Interpreter

    verif = {}
    for nombre, peso in [('pt', pt), ('fp32', TFLITE['fp32']), ('int8', TFLITE['int8'])]:
        m = YOLO(str(peso), task='detect').val(data=str(DATA_TEST_ALL), split='val', imgsz=640,
                                              batch=16 if nombre == 'pt' else 1, plots=False, verbose=False,
                                              project=str(RUNS_DIR), name=f'{ELEGIDO}_verif_{nombre}')
        verif[nombre] = {'mAP50': m.box.map50, 'mAP50-95': m.box.map,
                         **{f'mAP50-95 {CLASSES[c]}': m.box.class_result(i)[3] for i, c in enumerate(m.box.ap_class_index)},
                         'ms/img (GPU)' if nombre == 'pt' and torch.cuda.is_available() else 'ms/img (CPU)': m.speed['inference']}
    display(pd.DataFrame(verif).T.round(4))

    it = Interpreter(model_path=str(TFLITE['fp32']))
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    print('Entrada:', inp['shape'].tolist(), inp['dtype'].__name__, '| Salida:', out['shape'].tolist(), out['dtype'].__name__)

    # Decodificación como en la app: imagen -> 640x640 RGB float 0..1 -> salida [1, 6, 8400]
    ejemplo = sorted((DATA_DIR / 'test/images').iterdir())[0]
    img = cv2.cvtColor(cv2.imread(str(ejemplo)), cv2.COLOR_BGR2RGB)
    x = cv2.resize(img, (640, 640)).astype(np.float32)[None] / 255.0
    it.set_tensor(inp['index'], x)
    it.invoke()
    y = it.get_tensor(out['index'])[0]            # (6, 8400)
    scores = y[4:]                                # (2, 8400) ya en 0..1
    i = int(scores.max(axis=0).argmax())
    cx, cy, bw, bh = y[:4, i]
    print(f'{ejemplo.name}: clase {CLASSES[int(scores[:, i].argmax())]} conf {scores[:, i].max():.3f} '
          f'caja cx={cx:.3f} cy={cy:.3f} w={bw:.3f} h={bh:.3f} (normalizada 0..1)')
    print('Etiqueta real:', (DATA_DIR / 'test/labels' / f'{ejemplo.stem}.txt').read_text().strip())

    resumen_app = {
        'modelo': ELEGIDO, 'clases': {0: 'Crack', 1: 'Intact'}, 'conf_recomendado': CONF_APP,
        'entrada': {'shape': inp['shape'].tolist(), 'dtype': inp['dtype'].__name__, 'layout': 'NHWC RGB 0..1'},
        'salida': {'shape': out['shape'].tolist(), 'dtype': out['dtype'].__name__,
                   'filas': ['cx', 'cy', 'w', 'h', 'score_Crack', 'score_Intact']},
        'archivos_MB': {k: round(f.stat().st_size / 1e6, 2) for k, f in TFLITE.items()},
        'verificacion': {k: {m: round(float(v), 4) for m, v in d.items()} for k, d in verif.items()},
        'test_original_por_clase': comparacion[ELEGIDO].round(4).to_dict(),
        'acierto_por_grupo': grupos[ELEGIDO].round(4).to_dict(),
        'comparacion_v2': {'acierto_por_grupo_v2': grupos[BASE_RUN].round(4).to_dict(),
                           'test_original_por_clase_v2': comparacion[BASE_RUN].round(4).to_dict()},
        'commons': {n: {'acierto': round(float(d['correcto'].mean()), 4), 'n': len(d),
                        'por_clase': d.groupby('clase')['correcto'].mean().round(4).to_dict()} for n, d in commons.items()},
        'fotos_publicas': {'aceptadas': dict(pub_ok), 'totales': dict(pub_total), 'conf_ok': CONF_OK, 'conf_duda': CONF_DUDA},
    }
    (EXPORT_DIR / 'resumen.json').write_text(json.dumps(resumen_app, indent=2, ensure_ascii=False))
    print(json.dumps(resumen_app, indent=2, ensure_ascii=False))

## 11. Descargar los resultados
Empaqueta en `eggs_v3_resultados.zip` los modelos exportados (si ganó `v3`), la comparación `v2` vs `v3` y las hojas de auditoría, lo guarda en tu Drive y lo **descarga a tu PC** (carpeta Descargas). Si el navegador pregunta, permite la descarga.

`v3` tiene la misma entrada y salida que `v2`: la app solo cambia el nombre del archivo, y `dano_v1` sigue funcionando detrás sin cambios.

In [ ]:
from google.colab import files

paquete = Path('/content/eggs_v3_resultados')
if paquete.exists():
    shutil.rmtree(paquete)
paquete.mkdir()
shutil.copy2(RUNS_DIR / 'v3_comparacion.json', paquete)
shutil.copytree(AUD_DIR, paquete / 'auditoria')
if (EXPORTS_DIR / NEW_RUN).exists():
    shutil.copytree(EXPORTS_DIR / NEW_RUN, paquete / 'exports_v3')
for f in ['results.csv', 'results.png', 'args.yaml']:
    if (RUNS_DIR / NEW_RUN / f).exists():
        shutil.copy2(RUNS_DIR / NEW_RUN / f, paquete / f'entrenamiento_{f}')
zip_local = Path(shutil.make_archive('/content/eggs_v3_resultados', 'zip', paquete))
shutil.copy2(zip_local, OUT_DIR / zip_local.name)
print(f'Listo: {zip_local.name} ({zip_local.stat().st_size / 1e6:.1f} MB), copia en {OUT_DIR}')
files.download(str(zip_local))